# 08 - Ranking Model

The feature table from the previous notebook is a labeled (user, candidate product) dataset
— real supervised learning setup, a genuinely different approach from every model previous model,
all of which ranked products by hand-built heuristics rather than learning from labels
directly.

`HistGradientBoostingClassifier` is the model here — tree-based, so the mixed-scale numeric
features (counts, rates, order numbers) don't need pre-scaling, and it's in scikit-learn core
rather than pulling in XGBoost/LightGBM as an extra dependency for what a project this size
doesn't need. `class_weight='balanced'` accounts for the label skew found in feature
engineering (roughly 93/7 negative/positive on the full data).

The train/validation split is by `user_id`, not by row. Splitting rows randomly would let
the same user show up in both sets with different candidate products, which would let the
model pick up user-specific signal from train and get evaluated on more of that same user in
validation — an inflated score that wouldn't hold up on users the model has genuinely never
seen.

Logic in `src/ranking_model.py`.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd

from data_processing import load_raw_data
from eda import build_transactions
from baseline_models import PersonalizedFrequencyModel, get_eval_users, evaluate_model
from collaborative_filtering import ALSModel
from candidate_generation import generate_candidates_bulk
from feature_engineering import build_training_table, add_features
from ranking_model import RankingModel, split_by_user, FEATURE_COLUMNS

## Load the feature table

Reading straight from the parquet the previous notebook saved, rather than rebuilding
candidates and features from scratch.

In [2]:
processed_dir = Path.cwd().parent / "data" / "processed"
featured = pd.read_parquet(processed_dir / "ranking_training_table.parquet")
print(featured.shape)
print("positive rate:", featured["label"].mean())

(13511398, 15)
positive rate: 0.06618671139729583


The candidate sets themselves aren't saved to disk (they're products of the fitted PF/ALS
models, not raw data), so they're rebuilt here to pass into the ranking model — same models,
same settings as the previous notebook, so the candidates line up with the featured rows.

In [3]:
processed_path = Path.cwd().parent / "data" / "processed" / "transactions.parquet"
if processed_path.exists():
    txn = pd.read_parquet(processed_path)
else:
    data = load_raw_data()
    txn = build_transactions(data)

eval_users = get_eval_users(txn)
pf_model = PersonalizedFrequencyModel(top_n=50).fit(txn)
als_model = ALSModel(factors=50, regularization=0.01, alpha=15.0, iterations=15).fit(txn)
candidates = generate_candidates_bulk(eval_users, txn, pf_model, als_model, n_each=50)

C:\Users\shubh\AppData\Roaming\Python\Python314\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

## Split by user and train

In [4]:
train_table, val_table = split_by_user(featured, val_fraction=0.2)
print("train:", train_table.shape, train_table["user_id"].nunique(), "users")
print("val:  ", val_table.shape, val_table["user_id"].nunique(), "users")
print("train positive rate:", train_table["label"].mean())
print("val positive rate:  ", val_table["label"].mean())

train: (10806719, 15) 104968 users
val:   (2704679, 15) 26241 users
train positive rate: 0.06623703272010681
val positive rate:   0.06598564931365238


In [5]:
ranker = RankingModel(max_iter=100).fit(train_table, candidates)

## Feature importances

Worth checking before evaluating — if the model is leaning almost entirely on one feature
(likely `up_purchase_count`, given how dominant frequency was across every model in Phase 5),
that's useful context for reading the evaluation numbers, not a problem to fix.

In [6]:
from sklearn.inspection import permutation_importance

X_val = val_table[FEATURE_COLUMNS]
y_val = val_table["label"]

importances = permutation_importance(
    ranker.model, X_val, y_val, n_repeats=5, random_state=42, scoring="average_precision"
)

importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": importances.importances_mean,
}).sort_values("importance", ascending=False)

importance_df

,feature,importance
7,up_purchase_count,0.302292
11,orders_since_last_purchase,0.134115
0,n_orders,0.053911
3,reorder_rate,0.014212
5,product_reorder_rate,0.012165
4,n_purchases,0.002762
9,up_avg_cart_position,0.002286
6,avg_add_to_cart_position,0.002171
1,avg_basket_size,0.001185
8,up_last_order_number,0.000993


## Evaluate

Precision@10 / Recall@10, restricted to validation users only — evaluating on train users
would just measure how well the model fit data it already trained on. The featured
validation rows are grouped by user once up front so `recommend()` doesn't recompute
features per call.

In [7]:
val_lookup = {uid: grp for uid, grp in val_table.groupby("user_id")}
val_eval_users = eval_users[eval_users["user_id"].isin(val_lookup.keys())]

K = 10
ranking_results = evaluate_model(
    lambda uid: ranker.recommend(uid, val_lookup, n=K), val_eval_users, k=K
)
ranking_results

{'k': 10,
 'n_users_evaluated': 26241,
 'precision_at_k': 0.29526313783773483,
 'recall_at_k': 0.3501118018925534}

### Findings

The ranking-model phase moves from hand-crafted recommendation heuristics to a supervised learning approach. Using the feature table generated in the previous phase, the model learns to rank user-product candidates directly from labeled historical outcomes. A `HistGradientBoostingClassifier` is used with balanced class weighting, while the train/validation split is performed by **user_id** to prevent user-level information from leaking between training and validation sets.

* **The training setup preserves the strong class imbalance identified in Feature Engineering.** The final feature table contains **13,511,398 candidate pairs**, with a positive rate of **6.62%**. The model therefore uses `class_weight='balanced'` so that the relatively small positive class is not overwhelmed by negative examples.

* **The user-level split provides a more reliable validation design.** Approximately **104,968 users** are used for training and **26,241 users** for validation. The positive rates remain very similar at **6.62% in training** and **6.60% in validation**, indicating that the split preserves the underlying label distribution while keeping users completely separated between the two sets.

* **Purchase-history frequency remains the dominant predictive signal.** Permutation importance shows `up_purchase_count` as the most important feature with an importance score of **0.3023**, followed by `orders_since_last_purchase` at **0.1341** and `n_orders` at **0.0539**. This is consistent with the earlier EDA and baseline results showing that repeat purchase behavior is the strongest signal for predicting the next basket.

* **Recency adds meaningful information beyond purchase frequency.** `orders_since_last_purchase` is the second-most important feature, indicating that how recently a user purchased a product contributes materially to ranking performance in addition to the number of previous purchases.

* **The ranking model performs strongly on the validation subset.** Across **26,241 validation users**, the model achieves **Precision@10 = 0.2953** and **Recall@10 = 0.3501**. These results are above the previously observed Personalized Frequency performance of **0.2838 / 0.3298** and Hybrid performance of **0.2761 / 0.3303**; however, those earlier results were calculated on the full evaluation population, so the comparison should be treated as directional rather than a definitive improvement.

* **The model is learning and refining the same core behavioral signal rather than replacing it.** The high importance of `up_purchase_count`, together with the strong earlier performance of Personalized Frequency, suggests that the ranking model's advantage comes from combining purchase frequency with additional behavioral features such as recency and order history rather than discovering an entirely different recommendation mechanism.

* **Modeling implication:** The ranking model is a promising next-stage approach because it can learn how to combine the behavioral signals that were previously handled manually. The next step should be a **full evaluation against the same user population and evaluation protocol used for the baseline and hybrid models**, followed by comparison of whether the additional feature interactions provide a statistically meaningful improvement in recommendation quality.


In [8]:
pd.DataFrame([ranking_results], index=["Ranking Model"]).to_csv(
    processed_dir / "ranking_model_results.csv"
)